# 02. ETL C5 — Avance 03

**Proyecto A&C — Análisis de accidentes y hechos de tránsito en la Ciudad de México**  
**Equipo:** Manuel Soto Romero y Araceli Liliana Reyes Cabello · UACM · Inteligencia de Negocios · Grupo 313 · 2026-2

## Objetivo
Procesar los archivos de C5 de forma independiente. Se conservan los originales, se identifican duplicados exactos, folios repetidos, faltantes y posibles discrepancias territoriales. Cada registro recibe una clasificación primaria y se documenta en una bitácora.

**Pregunta de negocio:** ¿En qué alcaldías y franjas horarias de la Ciudad de México se registra la mayor cantidad de personas lesionadas en hechos de tránsito, de acuerdo con los registros disponibles a partir de 2018?

**Alcance:** C5 representa reportes de incidentes. Sus registros no se suman directamente con SSC o INEGI-ATUS. La hora de creación o cierre no sustituye la hora de ocurrencia. Las discrepancias territoriales solo se excluyen cuando existen dos campos comparables y la contradicción es comprobable.

**Entrada esperada:** archivos `.csv` en `data/raw/C5/`. Si los nombres reales de las columnas territoriales o de folio difieren, se revisan en el inventario antes de configurar los alias.

**Búsqueda de entrada:** además de `data/raw`, este notebook localiza archivos originales en `Datos/` y en carpetas identificadas del Avance 02.


## 1. Entorno, rutas y funciones auxiliares

Creo una carpeta por ejecución. Los resultados se guardan comprimidos (`.csv.gz`) para reducir espacio y evitar los errores de cuota observados durante las exportaciones anteriores.

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np
import hashlib
import re
import csv
import gzip
import unicodedata

TZ = ZoneInfo("America/Mexico_City")
from pathlib import Path

def localizar_raiz_proyecto():
    actual = Path.cwd().resolve()
    for carpeta in (actual, *actual.parents):
        if carpeta.name.lower() == "avance03":
            return carpeta
    raise FileNotFoundError(
        "No se encontró la carpeta Avance03 entre el directorio actual y sus carpetas superiores. "
        "Abre Jupyter desde Notebooks3/Avance03."
    )

ROOT = localizar_raiz_proyecto()
PROYECTO = ROOT.parent.parent
RAW = PROYECTO / "resultados" / "resultados_C5" / "etapa3"
ARCHIVO_C5 = RAW / "incidentes_viales_C5_etapa3.csv"
if not ARCHIVO_C5.is_file():
    raise FileNotFoundError(f"No se encontró el archivo C5 de etapa 3: {ARCHIVO_C5}")
ARCHIVOS = [ARCHIVO_C5]

RUN_ID = datetime.now(TZ).strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "data" / "processed" / "C5" / RUN_ID
LOG_DIR = ROOT / "bitacora" / "C5" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
BITACORA = []

# Configurar únicamente cuando el diccionario confirme los nombres exactos.
COL_ALCALDIA_1 = None
COL_ALCALDIA_2 = None


def norm(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"[^a-z0-9]+", "_", texto).strip("_")


def buscar_columna(df, aliases):
    columnas = {norm(c): c for c in df.columns}
    for alias in aliases:
        if norm(alias) in columnas:
            return columnas[norm(alias)]
    return None


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            digest.update(bloque)
    return digest.hexdigest()


def leer_csv_seguro(path):
    errores = []
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(
                path, dtype="string", encoding=encoding,
                sep=None, engine="python", on_bad_lines="error"
            )
        except Exception as error:
            errores.append(f"{encoding}: {error}")
    raise ValueError(
        f"No fue posible leer {path.name}. " + " | ".join(errores)
    )


def escribir(df, path):
    """Exporta comprimido y verifica el número de filas sin cargar todo de nuevo."""
    path = Path(path)
    if path.suffix != ".gz":
        path = Path(str(path) + ".gz")
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(
        path, index=False, encoding="utf-8-sig",
        compression="gzip"
    )
    with gzip.open(path, "rt", encoding="utf-8-sig", newline="") as archivo:
        lector = csv.reader(archivo)
        next(lector, None)
        filas = sum(1 for _ in lector)
    if filas != len(df):
        raise AssertionError(
            f"Exportación incompleta en {path.name}: {filas} != {len(df)}"
        )
    return path


def loggear(archivo, paso, regla, antes, despues, decision, justificacion, detalle=""):
    BITACORA.append({
        "fecha_hora_local": datetime.now(TZ).isoformat(timespec="seconds"),
        "fuente": "C5", "archivo_origen": str(archivo),
        "proceso": paso, "regla": regla,
        "registros_antes": int(antes), "registros_despues": int(despues),
        "decision": decision, "justificacion": justificacion,
        "detalle": detalle, "version_proceso": "Avance03-v3"
    })


print("Carpeta del proyecto:", ROOT)
print("Proyecto:", PROYECTO)
print("Archivo de entrada C5:", ARCHIVO_C5)
print("Archivos encontrados:", len(ARCHIVOS))
print("Salida de esta ejecución:", OUT)

## 2. Extracción e inventario

Cada archivo se mantiene como una tabla independiente. Registro sus dimensiones, columnas y huella SHA-256. No uno archivos con unidades de observación diferentes.

In [ ]:
inventario = []
tablas_originales = {}

for archivo in ARCHIVOS:
    df = leer_csv_seguro(archivo)
    tablas_originales[archivo.name] = df
    inventario.append({
        "archivo": archivo.name,
        "ruta": str(archivo),
        "registros": len(df),
        "columnas": len(df.columns),
        "sha256": sha256(archivo),
        "nombres_columnas": " | ".join(map(str, df.columns))
    })
    loggear(
        archivo.name, "Extracción", "RG-01/RG-11",
        len(df), len(df), "Conservar original y crear copia de trabajo",
        "El archivo original no se modifica; se registra su procedencia y huella."
    )

inventario_df = pd.DataFrame(inventario)
display(inventario_df)
escribir(inventario_df, OUT / "inventario_fuentes.csv.gz")

## 3. Duplicados exactos y folios repetidos

Se consideran duplicados exactos las filas que coinciden en todas las columnas originales. Se conserva la primera aparición y las copias adicionales se exportan por separado. Un folio repetido no se elimina automáticamente: se conserva para revisión.

In [ ]:
resultados = {}
resumen_calidad = []

ALIASES_FOLIO = [
    "folio", "numero_folio", "no_folio", "id_reporte",
    "id_incidente", "folio_c5", "numero_reporte"
]

for nombre, original in tablas_originales.items():
    entrada = len(original)

    # R01: duplicados exactos según todas las columnas originales.
    mascara_duplicados = original.duplicated(keep="first")
    duplicados = original.loc[mascara_duplicados].copy()
    base = original.loc[~mascara_duplicados].copy()

    if len(duplicados):
        duplicados["_motivo_separacion"] = "Copia exacta adicional"
        duplicados["_regla_aplicada"] = "C5-R01/RG-04"

    # R02: los folios repetidos se reportan, pero no se eliminan por sí solos.
    col_folio = buscar_columna(base, ALIASES_FOLIO)
    if col_folio:
        folios = base[col_folio].astype("string").str.strip()
        folios_validos = folios.notna() & folios.ne("")
        repetidos = folios_validos & folios.duplicated(keep=False)
        folios_revision = base.loc[repetidos].copy()
        if len(folios_revision):
            folios_revision["_motivo_revision"] = (
                "Folio repetido; se conserva porque no demuestra por sí solo duplicidad exacta"
            )
            folios_revision["_regla_aplicada"] = "C5-R02"
    else:
        folios_revision = base.iloc[0:0].copy()
        print(f"Aviso: {nombre}: no se identificó columna de folio; C5-R02 queda pendiente.")

    resultados[nombre] = {
        "base": base,
        "duplicados": duplicados,
        "folios_repetidos": folios_revision,
        "entrada": entrada,
        "col_folio": col_folio
    }

    resumen_calidad.append({
        "archivo": nombre,
        "registros_iniciales": entrada,
        "duplicados_exactos_retirados": len(duplicados),
        "registros_sin_copias_exactas": len(base),
        "filas_con_folio_repetido_revision": len(folios_revision),
        "columna_folio": col_folio
    })

    loggear(
        nombre, "Duplicados exactos", "C5-R01/RG-04",
        entrada, len(base), "Retirar copias exactas adicionales",
        "Se conserva la primera aparición y se exporta evidencia de las copias adicionales."
    )
    loggear(
        nombre, "Folios repetidos", "C5-R02/RG-05",
        len(base), len(base), "Conservar y marcar para revisión",
        "La repetición del folio no basta para eliminar registros diferentes.",
        f"Filas incluidas en revisión: {len(folios_revision)}"
    )

resumen_calidad_df = pd.DataFrame(resumen_calidad)
display(resumen_calidad_df)
escribir(resumen_calidad_df, OUT / "resumen_calidad_inicial.csv.gz")

## 4. Reglas territoriales y temporales

Las reglas se aplican solo cuando se identifican columnas válidas. La ausencia de hora de ocurrencia deja el registro pendiente para análisis temporal, pero no lo elimina de otros análisis. La hora de creación o cierre nunca se usa como sustituto.

In [ ]:
for nombre, resultado in resultados.items():
    base = resultado["base"].copy()
    base["_decision"] = "conservado"
    base["_motivo_decision"] = "Sin exclusión automática comprobada; revisar variables no mapeadas"
    base["_regla_aplicada"] = "RG-10"
    base["_valores_ausentes"] = base.isna().sum(axis=1).astype("int64")

    # Identificar SD y vacíos sin confundirlos con el valor numérico cero.
    base["_campos_sd"] = 0
    base["_campos_vacios"] = 0
    for columna in base.columns:
        if pd.api.types.is_string_dtype(base[columna].dtype):
            valores = base[columna].fillna("").astype("string").str.strip()
            base["_campos_sd"] += valores.str.upper().isin(
                ["SD", "S/D", "NO DISPONIBLE"]
            ).astype("int64")
            base["_campos_vacios"] += valores.eq("").astype("int64")

    # C5-R03: comparar solo dos columnas confirmadas en el diccionario.
    if COL_ALCALDIA_1 and COL_ALCALDIA_2:
        if COL_ALCALDIA_1 in base.columns and COL_ALCALDIA_2 in base.columns:
            a = base[COL_ALCALDIA_1].fillna("").astype("string").str.strip().str.upper()
            b = base[COL_ALCALDIA_2].fillna("").astype("string").str.strip().str.upper()
            conflicto = a.ne("") & b.ne("") & a.ne(b)
            base.loc[conflicto, "_decision"] = "excluido"
            base.loc[conflicto, "_motivo_decision"] = (
                f"Discrepancia territorial comprobada entre {COL_ALCALDIA_1} y {COL_ALCALDIA_2}"
            )
            base.loc[conflicto, "_regla_aplicada"] = "C5-R03"
            discrepancias = base.loc[conflicto].copy()
            loggear(
                nombre, "Validación territorial", "C5-R03",
                len(base), len(base) - int(conflicto.sum()),
                "Excluir contradicciones territoriales comprobadas",
                "No se asignan alcaldías mediante suposiciones.",
                f"Discrepancias detectadas: {int(conflicto.sum())}"
            )
        else:
            discrepancias = base.iloc[0:0].copy()
            print(
                f"Aviso: {nombre}: los campos territoriales configurados no existen. "
                "C5-R03 queda pendiente."
            )
    else:
        discrepancias = base.iloc[0:0].copy()
        print(
            f"{nombre}: C5-R03 pendiente. Configura COL_ALCALDIA_1 y "
            "COL_ALCALDIA_2 solo después de confirmar el diccionario."
        )

    # C5-R04/R05: se crea una evidencia temporal superpuesta, sin sacar filas
    # de la tabla general que puede servir para otros análisis.
    col_hora = buscar_columna(
        base, ["hora_ocurrencia", "hora_del_incidente", "hora_accidente", "hora_hecho"]
    )
    if col_hora:
        hora = base[col_hora].fillna("").astype("string").str.strip()
        sin_hora = hora.isna() | hora.eq("") | hora.str.upper().isin(["SD", "S/D"])
        pendientes_temporales = base.loc[sin_hora].copy()
        pendientes_temporales["_motivo_revision"] = (
            "Sin hora de ocurrencia: se conserva para otros análisis, no para franjas horarias"
        )
        pendientes_temporales["_regla_aplicada"] = "C5-R04/C5-R05"
    else:
        pendientes_temporales = base.iloc[0:0].copy()
        print(
            f"{nombre}: no se identificó hora de ocurrencia. "
            "No se utilizará hora de creación/cierre como sustituto."
        )

    resultado["base"] = base
    resultado["discrepancias"] = discrepancias
    resultado["pendientes_temporales"] = pendientes_temporales
    resultado["col_hora_ocurrencia"] = col_hora

    loggear(
        nombre, "Revisión temporal", "C5-R04/C5-R05",
        len(base), len(base), "Conservar sin hora para otros análisis",
        "La falta de hora solo limita el análisis temporal; no elimina el reporte general.",
        f"Filas pendientes para franjas horarias: {len(pendientes_temporales)}"
    )

## 5. Exportación, conciliación y bitácora

La conciliación utiliza categorías primarias mutuamente excluyentes: conservados + pendientes + excluidos + duplicados exactos retirados = registros iniciales. Los folios repetidos y los registros sin hora son reportes de revisión superpuestos; no se suman nuevamente.

In [ ]:
controles = []

for nombre, resultado in resultados.items():
    base = resultado["base"]
    duplicados = resultado["duplicados"]
    folios_revision = resultado["folios_repetidos"]
    discrepancias = resultado["discrepancias"]
    pendientes_temporales = resultado["pendientes_temporales"]

    conservados = base.loc[base["_decision"].eq("conservado")].copy()
    pendientes = base.loc[base["_decision"].eq("pendiente")].copy()
    excluidos = base.loc[base["_decision"].eq("excluido")].copy()

    # Se exportan todos los productos comprimidos para limitar uso de cuota.
    escribir(conservados, OUT / f"{Path(nombre).stem}_limpio.csv.gz")
    escribir(duplicados, OUT / f"{Path(nombre).stem}_duplicados_exactos.csv.gz")
    escribir(folios_revision, OUT / f"{Path(nombre).stem}_folios_repetidos_revision.csv.gz")
    escribir(pendientes, OUT / f"{Path(nombre).stem}_pendientes.csv.gz")
    escribir(excluidos, OUT / f"{Path(nombre).stem}_excluidos_por_regla.csv.gz")
    escribir(discrepancias, OUT / f"{Path(nombre).stem}_discrepancias_territoriales.csv.gz")
    escribir(pendientes_temporales, OUT / f"{Path(nombre).stem}_sin_hora_ocurrencia_revision_temporal.csv.gz")

    total = len(conservados) + len(pendientes) + len(excluidos) + len(duplicados)
    correcto = total == resultado["entrada"]
    controles.append({
        "archivo": nombre,
        "registros_iniciales": resultado["entrada"],
        "duplicados_exactos_retirados": len(duplicados),
        "conservados": len(conservados),
        "pendientes_primarios": len(pendientes),
        "excluidos": len(excluidos),
        "folios_repetidos_revision_superpuesta": len(folios_revision),
        "sin_hora_revision_temporal_superpuesta": len(pendientes_temporales),
        "total_reconciliado": total,
        "conciliacion_correcta": correcto
    })

    if not correcto:
        raise AssertionError(
            f"Falla de conciliación en {nombre}: {total} != {resultado['entrada']}"
        )

    loggear(
        nombre, "Conciliación", "RG-07/RG-11/RG-12",
        resultado["entrada"], total, "Conciliación correcta",
        "Las categorías primarias son excluyentes y suman el total inicial."
    )

control_df = pd.DataFrame(controles)
display(control_df)
escribir(control_df, OUT / "conciliacion_por_archivo.csv.gz")

bitacora_df = pd.DataFrame(BITACORA)
escribir(bitacora_df, LOG_DIR / "bitacora_C5.csv.gz")
escribir(bitacora_df, OUT / "bitacora_C5.csv.gz")

print()
print("ETL C5 terminado.")
print("Resultados:", OUT)
print("Bitácora:", LOG_DIR / "bitacora_C5.csv.gz")

## 6. Interpretación y limitaciones

Los conteos definitivos son los obtenidos al ejecutar con los CSV originales de C5. Los valores documentados en el Avance 02 son antecedentes que deben comprobarse nuevamente. Si C5-R03 no tiene configurados dos campos territoriales comparables, la regla queda expresamente pendiente y no se excluyen filas por suposición. La falta de hora de ocurrencia limita el análisis por franjas, pero no invalida automáticamente el registro para otros análisis.